# Publication tilt census

A gallery of full-page scientific figures describing **tilt magnitude and direction**, always separating anticyclonic (AE) and cyclonic (CE) eddies. Counts are deliberately secondary. Daily distributions use equal total weight per eddy so long tracks do not dominate.

The figures progress from the full population to region, subregion, lifecycle and lifetime-class contrasts. Set SAVE_FIGURES=True to export vector PDFs and 600 dpi PNGs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import seacofs_tilt_tools as tilt

AE_COLOUR, CE_COLOUR = '#C4473A', '#247BA0'
TYPE_COLOURS = {'AE': AE_COLOUR, 'CE': CE_COLOUR}
REGION_ORDER = ['Shelf', 'Upstream', 'Downstream']
SUBREGION_ORDER = ['S1', 'S2', 'U1', 'U2', 'D1', 'D2']
DIRECTION_MIN_TILT_KM = 5.0
LIFETIME_QUANTILES = (1/3, 2/3)
SAVE_FIGURES = False
FIGURE_DIR = Path('tilt_census_figures')
mpl.rcParams.update({'figure.dpi':120, 'savefig.dpi':600, 'font.size':10,
 'axes.titlesize':12, 'axes.labelsize':10, 'legend.fontsize':9,
 'axes.spines.top':False, 'axes.spines.right':False, 'pdf.fonttype':42, 'ps.fonttype':42})

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
region_group_map = {'S1':'Shelf','S2':'Shelf','U1':'Upstream','U2':'Upstream','D1':'Downstream','D2':'Downstream'}
df_eddies['RegionGroup'] = df_eddies['Region'].map(region_group_map)
df_eddies = df_eddies.sort_values(['Cyc','Eddy','Day']).copy()
keys = ['Cyc','Eddy']
df_eddies['day_index'] = df_eddies.groupby(keys).cumcount()
last_index = df_eddies.groupby(keys)['day_index'].transform('max')
df_eddies['norm_time'] = np.where(last_index > 0, df_eddies['day_index']/last_index, np.nan)
df_eddies['lifetime_days'] = df_eddies.groupby(keys)['Day'].transform(lambda x: x.max()-x.min()+1)
tilt_data = df_eddies.dropna(subset=['TiltDis','Cyc']).query("Cyc in ['AE','CE']").copy()
direction_data = tilt_data.dropna(subset=['TiltDir']).query('TiltDis >= @DIRECTION_MIN_TILT_KM').copy()

def equal_eddy_weights(data):
    return 1.0/data.groupby(keys)['Eddy'].transform('size').astype(float)

display(pd.DataFrame({cyc:{
 'tilt observations':int((tilt_data.Cyc==cyc).sum()),
 'eddies represented':int(tilt_data.loc[tilt_data.Cyc==cyc,'Eddy'].nunique()),
 'median tilt (km)':tilt_data.loc[tilt_data.Cyc==cyc,'TiltDis'].median()}
 for cyc in ['AE','CE']}).T.round(2))

In [ ]:
def panel_label(ax, label):
    ax.text(-.10, 1.05, label, transform=ax.transAxes, weight='bold', fontsize=12)

def finish_figure(fig, name):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURE_DIR/f'{name}.pdf', bbox_inches='tight')
        fig.savefig(FIGURE_DIR/f'{name}.png', bbox_inches='tight', dpi=600)
    plt.show()

def weighted_ecdf(values, weights):
    values, weights = np.asarray(values), np.asarray(weights)
    ok = np.isfinite(values)&np.isfinite(weights)&(weights>0)
    order = np.argsort(values[ok]); x, w = values[ok][order], weights[ok][order]
    return x, np.cumsum(w)/w.sum()

def weighted_quantile(values, quantiles, weights):
    values, weights = np.asarray(values), np.asarray(weights)
    ok = np.isfinite(values)&np.isfinite(weights)&(weights>0)
    order = np.argsort(values[ok]); x, w = values[ok][order], weights[ok][order]
    cdf = (np.cumsum(w)-.5*w)/w.sum()
    return np.interp(quantiles, cdf, x)

def circular_summary_deg(theta, weights=None):
    theta = np.asarray(theta,float); weights = np.ones(theta.size) if weights is None else np.asarray(weights,float)
    ok = np.isfinite(theta)&np.isfinite(weights)&(weights>0)
    if not ok.any(): return np.nan, np.nan
    z = np.average(np.exp(1j*np.deg2rad(theta[ok])), weights=weights[ok])
    return np.rad2deg(np.angle(z))%360, np.abs(z)

def polar_hist(ax, data, colour, bins=np.arange(0,361,15)):
    theta = np.deg2rad(data['TiltDir'].to_numpy()); w = equal_eddy_weights(data).to_numpy()
    hist, edges = np.histogram(theta, bins=np.deg2rad(bins), weights=w, density=True)
    ax.bar(edges[:-1], hist, width=np.diff(edges), align='edge', color=colour,
           edgecolor='white', linewidth=.5, alpha=.82)
    mean, concentration = circular_summary_deg(data['TiltDir'], w)
    ymax = hist.max() if hist.size else 1
    ax.annotate('', xy=(np.deg2rad(mean), ymax*concentration), xytext=(0,0),
                arrowprops=dict(color='black', width=1.6, headwidth=7))
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1); ax.set_yticklabels([]); ax.grid(alpha=.25)
    return mean, concentration

def binned_tilt_summary(data, xcol, edges):
    work=data.dropna(subset=[xcol,'TiltDis']).copy(); work['_bin']=pd.cut(work[xcol],edges,include_lowest=True)
    rows=[]
    for (cyc,b),g in work.groupby(['Cyc','_bin'],observed=True):
        q25,med,q75=weighted_quantile(g.TiltDis,[.25,.5,.75],equal_eddy_weights(g)); rows.append((cyc,b.mid,q25,med,q75))
    return pd.DataFrame(rows,columns=['Cyc','x','q25','median','q75'])

def binned_direction_summary(data, xcol, edges):
    work=data.dropna(subset=[xcol,'TiltDir']).copy(); work['_bin']=pd.cut(work[xcol],edges,include_lowest=True)
    rows=[]
    for (cyc,b),g in work.groupby(['Cyc','_bin'],observed=True):
        mean,r=circular_summary_deg(g.TiltDir,equal_eddy_weights(g)); rows.append((cyc,b.mid,mean,r))
    return pd.DataFrame(rows,columns=['Cyc','x','mean_direction','concentration'])

## Figure 1 — Full-population tilt-magnitude distribution
Equal-eddy-weighted density and exceedance probability reveal typical tilt and the extreme tail.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11.7,5.8),constrained_layout=True)
upper=tilt_data.TiltDis.quantile(.995); bins=np.linspace(0,upper,55)
for cyc in ['AE','CE']:
    g=tilt_data[tilt_data.Cyc==cyc]; w=equal_eddy_weights(g)
    axes[0].hist(g.TiltDis,bins=bins,weights=w/w.sum(),histtype='stepfilled',alpha=.30,color=TYPE_COLOURS[cyc],label=cyc)
    axes[0].hist(g.TiltDis,bins=bins,weights=w/w.sum(),histtype='step',lw=2,color=TYPE_COLOURS[cyc])
    x,f=weighted_ecdf(g.TiltDis,w); axes[1].plot(x,1-f,lw=2.4,color=TYPE_COLOURS[cyc],label=cyc)
axes[0].set(xlabel='Tilt distance (km)',ylabel='Weighted probability per bin',xlim=(0,upper))
axes[1].set(xlabel='Tilt distance (km)',ylabel='Exceedance probability',xlim=(0,upper),yscale='log')
for i,ax in enumerate(axes): panel_label(ax,chr(97+i)); ax.grid(alpha=.18); ax.legend(frameon=False)
fig.suptitle('Eddy tilt magnitude: full-population distribution'); finish_figure(fig,'01_overall_magnitude')

## Figure 2 — Regional tilt-magnitude distributions
Side-by-side violins compare Shelf, Upstream and Downstream while preserving the full distribution.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(11.7,6.2),sharey=True,constrained_layout=True)
for j,(ax,region) in enumerate(zip(axes,REGION_ORDER)):
    arrays=[tilt_data.loc[(tilt_data.Cyc==cyc)&(tilt_data.RegionGroup==region)].groupby(keys).TiltDis.median().dropna() for cyc in ['AE','CE']]
    parts=ax.violinplot(arrays,positions=[1,2],showextrema=False,showmedians=True,widths=.8)
    for body,cyc in zip(parts['bodies'],['AE','CE']): body.set_facecolor(TYPE_COLOURS[cyc]); body.set_edgecolor(TYPE_COLOURS[cyc]); body.set_alpha(.35)
    parts['cmedians'].set_color('black'); ax.set_xticks([1,2],['AE','CE']); ax.set_title(region)
    ax.set_ylim(0,tilt_data.TiltDis.quantile(.99)); ax.grid(axis='y',alpha=.2); panel_label(ax,chr(97+j))
axes[0].set_ylabel('Tilt distance (km)'); fig.suptitle('Regional distributions of tilt magnitude')
finish_figure(fig,'02_region_magnitude_violins')

## Figure 3 — Six-subregion magnitude atlas
Weighted median, interquartile range and 10–90% interval for S1–D2.

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(8.3,10.5),sharex=True,constrained_layout=True)
for ax,cyc in zip(axes,['AE','CE']):
    rows=[]
    for region in SUBREGION_ORDER:
        g=tilt_data[(tilt_data.Cyc==cyc)&(tilt_data.Region==region)]
        rows.append(weighted_quantile(g.TiltDis,[.1,.25,.5,.75,.9],equal_eddy_weights(g)))
    q=np.asarray(rows); x=np.arange(6)
    ax.vlines(x,q[:,0],q[:,4],color=TYPE_COLOURS[cyc],alpha=.35,lw=3)
    ax.vlines(x,q[:,1],q[:,3],color=TYPE_COLOURS[cyc],lw=9)
    ax.scatter(x,q[:,2],s=50,facecolor='white',edgecolor=TYPE_COLOURS[cyc],zorder=3)
    ax.set(title=cyc,ylabel='Tilt distance (km)'); ax.grid(axis='y',alpha=.2)
axes[-1].set_xticks(np.arange(6),SUBREGION_ORDER); fig.suptitle('Tilt-magnitude contrasts across six EAC subregions')
finish_figure(fig,'03_subregion_magnitude_atlas')

## Figure 4 — Full-population tilt-direction census
Arrows show circular mean direction and resultant concentration.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11.7,6.2),subplot_kw={'projection':'polar'},constrained_layout=True)
for ax,cyc,label in zip(axes,['AE','CE'],['a','b']):
    g=direction_data[direction_data.Cyc==cyc]; mean,r=polar_hist(ax,g,TYPE_COLOURS[cyc])
    ax.set_title(f'{cyc}\nmean={mean:.0f}°, concentration={r:.2f}',pad=18); panel_label(ax,label)
fig.suptitle(f'Tilt direction for displacements ≥ {DIRECTION_MIN_TILT_KM:g} km')
finish_figure(fig,'04_overall_direction')

## Figure 5 — Regional direction atlas
Six polar panels test whether the AE–CE contrast persists across Shelf, Upstream and Downstream.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(11.7,8.3),subplot_kw={'projection':'polar'},constrained_layout=True)
for i,cyc in enumerate(['AE','CE']):
    for j,region in enumerate(REGION_ORDER):
        ax=axes[i,j]; g=direction_data[(direction_data.Cyc==cyc)&(direction_data.RegionGroup==region)]
        mean,r=polar_hist(ax,g,TYPE_COLOURS[cyc]); ax.set_title(f'{cyc} — {region}\nμ={mean:.0f}°, R={r:.2f}',fontsize=10,pad=12)
        panel_label(ax,chr(97+i*3+j))
fig.suptitle('Regional structure of tilt direction'); finish_figure(fig,'05_region_direction_atlas')

## Figure 6 — Joint magnitude–direction fingerprint
Polar two-dimensional histograms show where directional preference and large displacement coexist.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11.7,6.4),subplot_kw={'projection':'polar'},constrained_layout=True)
theta_edges=np.deg2rad(np.arange(0,361,15)); radial_edges=np.linspace(0,direction_data.TiltDis.quantile(.99),28)
histograms={}
for cyc in ['AE','CE']:
    g=direction_data[direction_data.Cyc==cyc]
    histograms[cyc]=np.histogram2d(np.deg2rad(g.TiltDir),g.TiltDis,bins=[theta_edges,radial_edges],weights=equal_eddy_weights(g))[0]
positive=np.concatenate([H[H>0] for H in histograms.values()]); norm=LogNorm(vmin=max(positive.min(),1e-4),vmax=max(H.max() for H in histograms.values()))
for ax,cyc,label in zip(axes,['AE','CE'],['a','b']):
    mesh=ax.pcolormesh(theta_edges,radial_edges,histograms[cyc].T,cmap='magma',norm=norm,shading='flat')
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1); ax.set_title(cyc,pad=18); panel_label(ax,label); ax.set_rlabel_position(225)
fig.colorbar(mesh,ax=axes,shrink=.72,label='Equal-eddy-weighted frequency')
fig.suptitle('Joint fingerprint of tilt distance and bearing'); finish_figure(fig,'06_joint_magnitude_direction')

## Figure 7 — Tilt magnitude through the eddy lifecycle
Weighted median and interquartile range in normalised-lifetime bins.

In [ ]:
life_edges=np.linspace(0,1,16); summary=binned_tilt_summary(tilt_data,'norm_time',life_edges)
fig,ax=plt.subplots(figsize=(10.5,7.2),constrained_layout=True)
for cyc in ['AE','CE']:
    s=summary[summary.Cyc==cyc]; ax.fill_between(s.x,s.q25,s.q75,color=TYPE_COLOURS[cyc],alpha=.18)
    ax.plot(s.x,s['median'],'-o',color=TYPE_COLOURS[cyc],lw=2.4,ms=4,label=cyc)
ax.set(xlabel='Normalised eddy lifetime',ylabel='Tilt distance (km)',title='Evolution of tilt magnitude through the eddy lifecycle',xlim=(0,1))
ax.grid(alpha=.2); ax.legend(frameon=False); finish_figure(fig,'07_lifecycle_magnitude')

## Figure 8 — Direction and directional coherence through life
Mean bearing is paired with resultant concentration so weakly defined directions are visible.

In [ ]:
summary=binned_direction_summary(direction_data,'norm_time',life_edges)
fig,axes=plt.subplots(2,1,figsize=(8.3,10.5),sharex=True,constrained_layout=True)
for cyc in ['AE','CE']:
    s=summary[summary.Cyc==cyc]
    axes[0].plot(s.x,s.mean_direction,'-o',color=TYPE_COLOURS[cyc],lw=2,ms=4,label=cyc)
    axes[1].plot(s.x,s.concentration,'-o',color=TYPE_COLOURS[cyc],lw=2,ms=4,label=cyc)
axes[0].set(ylabel='Circular-mean direction (°)',yticks=[0,90,180,270,360],ylim=(0,360))
axes[1].set(xlabel='Normalised eddy lifetime',ylabel='Resultant concentration',ylim=(0,1),xlim=(0,1))
for i,ax in enumerate(axes): ax.grid(alpha=.2); panel_label(ax,chr(97+i)); ax.legend(frameon=False)
fig.suptitle('Evolution of tilt direction and directional coherence'); finish_figure(fig,'08_lifecycle_direction')

## Figure 9 — Short-, intermediate- and long-lived eddies
Lifetime classes use eddy-level terciles calculated separately within AE and CE.

In [ ]:
eddy_life=tilt_data.groupby(keys,as_index=False)['lifetime_days'].max(); labels=['Short-lived','Intermediate','Long-lived']; maps=[]
for cyc,g in eddy_life.groupby('Cyc'):
    q1,q2=g.lifetime_days.quantile(LIFETIME_QUANTILES); g=g.copy()
    g['LifetimeClass']=pd.cut(g.lifetime_days,[-np.inf,q1,q2,np.inf],labels=labels); maps.append(g)
life_map=pd.concat(maps,ignore_index=True); life_data=tilt_data.merge(life_map,on=keys+['lifetime_days'],how='left')
fig,axes=plt.subplots(1,2,figsize=(11.7,6.2),sharey=True,constrained_layout=True)
for ax,cyc,panel in zip(axes,['AE','CE'],['a','b']):
    arrays=[life_data.loc[(life_data.Cyc==cyc)&(life_data.LifetimeClass==c)].groupby(keys).TiltDis.median().dropna() for c in labels]
    parts=ax.violinplot(arrays,showextrema=False,showmedians=True)
    for body in parts['bodies']: body.set_facecolor(TYPE_COLOURS[cyc]); body.set_alpha(.35)
    parts['cmedians'].set_color('black'); ax.set_xticks([1,2,3],labels,rotation=18); ax.set(title=cyc,ylabel='Tilt distance (km)')
    ax.set_ylim(0,life_data.TiltDis.quantile(.99)); ax.grid(axis='y',alpha=.2); panel_label(ax,panel)
fig.suptitle('Tilt magnitude by eddy lifetime class'); finish_figure(fig,'09_lifetime_class_magnitude')

## Figure 10 — Direction by lifetime class
Tests whether directional preference is concentrated in long-lived coherent eddies.

In [ ]:
life_direction=direction_data.merge(life_map,on=keys+['lifetime_days'],how='left')
fig,axes=plt.subplots(2,3,figsize=(11.7,8.3),subplot_kw={'projection':'polar'},constrained_layout=True)
for i,cyc in enumerate(['AE','CE']):
    for j,cls in enumerate(labels):
        ax=axes[i,j]; g=life_direction[(life_direction.Cyc==cyc)&(life_direction.LifetimeClass==cls)]
        mean,r=polar_hist(ax,g,TYPE_COLOURS[cyc]); ax.set_title(f'{cyc} — {cls}\nμ={mean:.0f}°, R={r:.2f}',fontsize=10,pad=12)
        panel_label(ax,chr(97+i*3+j))
fig.suptitle('Tilt direction by eddy lifetime class'); finish_figure(fig,'10_lifetime_class_direction')

## Figure selection notes

- These are candidate full-page figures, not ten figures that must all enter the paper.
- Figures 1, 4, 5, 6 and 7 provide the strongest compact census narrative.
- Rerun direction figures at 10 and 20 km as threshold-sensitivity tests.
- A mean direction is not meaningful when resultant concentration is close to zero.
- Before submission, bootstrap uncertainty by eddy for medians, quantiles and circular summaries.